In [1]:
import pandas as pd
from datetime import datetime, timedelta
from pprint import pprint
import dynamo # custom functions I wrote

def read_local(table_name, refresh):
    '''looks for pickled list of records locally 
    this can be swapped out with a function that loads external data 
    or a function that checks for local data before loading external data'''
    import pickle
         
    def read_local_file():
        with open("data/" + table_name, "rb") as file:
           data = pickle.load(file)
        return data
        
    if refresh:
        dynamo.refresh_table(table_name)
        
    else:
        print('Warning! Data may not be up to date')
    
    return read_local_file()


def save_csv(df, name):
    '''given a dataframe and a table_name, this function saves the dataframe as a csv 
    it is saved in a data folder in the current directory named yyyy-mm-dd_<table-name-here>
    this is super brittle. folder must exist and follows unix directory conventions'''
    date = str(datetime.now().date())    
    file_name = f'data/{date}_{name}.csv'
    print(f'saving at {file_name}')
    df.to_csv(file_name)
    
    
def append_df(new_row, df):
    df = pd.concat([df, new_row], ignore_index=True)
    return df


def format_time(timestamp):
    '''turns unix timestamp into Python DateTime object'''
    return datetime.fromtimestamp(timestamp)


rentals = read_local('rentals', refresh=False)

Warning! Data may not be up to date


In [2]:
def zoiby_stats():
    '''this is just a really lightweight function that Zoiby threw together to scan thrrough each record
    I use it as a sanity check to make sure my numbers match when I'm troubleshooting data access issues'''
    client = dynamo.connect()
    response = client.scan(
        TableName='rentals'
    )
    data = response['Items']

    while 'LastEvaluatedKey' in response:
        response = client.scan(
            TableName='rentals',
            ExclusiveStartKey=response['LastEvaluatedKey']
        )
        data.extend(response['Items'])
    total_rentals = 0
    total_rentals_bg = {
        'common': 0,
        'rare': 0,
        'meta': 0,
        'legendary': 0
    }
    active_rentals = 0
    active_rentals_bg = {
        'common': 0,
        'rare': 0,
        'meta': 0,
        'legendary': 0
    }
    rentals_in_daily_status = 0
    daily_rentals_bg = {
        'common': 0,
        'rare': 0,
        'meta': 0,
        'legendary': 0
    }

    for item in data:
        item = dynamo.get_py_dict(item)
        total_rentals += 1
        total_rentals_bg[item['degen']['background']] += 1
        if item['is_active'] is True:
            active_rentals += 1
            active_rentals_bg[item['degen']['background']] += 1
        if item['is_daily'] is True:
            rentals_in_daily_status += 1
            daily_rentals_bg[item['degen']['background']] += 1

    print(f'Total Rentals:\t{total_rentals}')
    print(f'Total Active Rentals:\t{active_rentals}')
    print(f'Total Daily Rentals:\t{rentals_in_daily_status}')
    print(f'Total Rentals by BG:\t{total_rentals_bg}')
    print(f'Active Rentals by BG:\t{active_rentals_bg}')
    print(f'Daily Rentals by BG:\t{daily_rentals_bg}')
    
zoiby_stats()

Total Rentals:	4199
Total Active Rentals:	1564
Total Daily Rentals:	1033
Total Rentals by BG:	{'common': 1598, 'rare': 952, 'meta': 1035, 'legendary': 614}
Active Rentals by BG:	{'common': 377, 'rare': 437, 'meta': 430, 'legendary': 320}
Daily Rentals by BG:	{'common': 356, 'rare': 194, 'meta': 285, 'legendary': 198}


In [3]:
def build_rentals_table(rentals):
    
    df = pd.DataFrame(columns = ['id', 'created_at', 'ended_at', 'user_id', 'renter_id', 'starting_transaction', 'is_sponsored', 'is_active', 'gross_cost', 'earnings', 'earning_cap', 'wins', 'bg', 'matches', 'ended_reason'])
    # to do: add in the logic to find the degen's owner from one of the transaction tables 
    
    for rental in rentals:
        rental_id = rental['id']
        created_at = format_time(rental['created_at']).date()
        user_id = rental['user_id']
        is_sponsored = (rental['user_id'] != rental['renter_id']) # if user doesn't match renter, returns true
        is_active = rental['is_active']
        bg = rental['degen']['background']
        earning_cap = rental['earning_cap']

        try:
            ended_reason = rental['ended_reason']
        except KeyError:
            ended_reason = None

        try:
            matches = rental['stats']['total']['matches']
        except KeyError:
            matches = 0

        try:
            wins = rental['stats']['total']['wins']
        except KeyError:
            wins = 0

        try:
            cost = rental['stats']['total']['charges']
        except KeyError:
            cost = 0

        if is_active:
            ended_at = None
        else:
            ended_at = format_time(rental['ended_at']).date()

        try:
            earnings = rental['stats']['total']['earnings']
        except KeyError:
            earnings = 0

        df_dict = {'id' : rental_id,
                   'created_at' : created_at,
                   'ended_at' : ended_at,
                   'user_id' : user_id,
                   'renter_id' : rental['renter_id'],
                   'starting_transaction' : rental['charge_txs'][0],
                   'is_sponsored' : is_sponsored,
                   'is_active' : is_active,
                   'gross_cost': cost,
                   'earnings' : earnings,
                   'earning_cap' : earning_cap,
                   'wins': wins,
                   'rarity' : bg,
                   'matches' : matches,
                   'ended_reason' : ended_reason}        
        new_row = pd.DataFrame([df_dict])
        df = append_df(new_row, df)
    return df

rentals_table = build_rentals_table(rentals)
rentals_table.tail()

,id,created_at,ended_at,user_id,renter_id,starting_transaction,is_sponsored,is_active,gross_cost,earnings,earning_cap,wins,bg,matches,ended_reason,rarity
4096,7816#us-east-1:298f8ca0-4049-4ad1-8237-9ee5f38...,2022-05-09,2022-05-16,us-east-1:fcb99144-fa00-48b8-8deb-4e04e89cbc92,us-east-1:298f8ca0-4049-4ad1-8237-9ee5f38ac564,us-east-1:298f8ca0-4049-4ad1-8237-9ee5f38ac564...,True,False,1000,202.6224,4004,1,NaN,39,Manually terminated previously,common
4097,4976#us-east-1:ad3a332d-ef16-4365-89a7-ea8aaf8...,2022-05-24,None,us-east-1:a4738958-80f1-42ec-88da-cd62b526567f,us-east-1:ad3a332d-ef16-4365-89a7-ea8aaf859a63,us-east-1:ad3a332d-ef16-4365-89a7-ea8aaf859a63...,True,True,1500,3134.7572,6006,2,NaN,161,None,rare
4098,5305#us-east-1:c941c2d5-8204-4169-ab3c-de4e37c...,2022-05-23,None,us-east-1:c941c2d5-8204-4169-ab3c-de4e37cd499a,us-east-1:c941c2d5-8204-4169-ab3c-de4e37cd499a,us-east-1:c941c2d5-8204-4169-ab3c-de4e37cd499a...,False,True,2000,917.6500,8008,9,NaN,15,None,meta
4099,3403#us-east-1:0c90631a-5ac7-435c-95a7-8f4120f...,2022-05-25,None,us-east-1:0c90631a-5ac7-435c-95a7-8f4120ffb62a,us-east-1:0c90631a-5ac7-435c-95a7-8f4120ffb62a,us-east-1:0c90631a-5ac7-435c-95a7-8f4120ffb62a...,False,True,7552,134.4320,14014,0,NaN,3,None,legendary
4100,8324#us-east-1:55ce4aec-2a3e-4596-b47a-8cd4b2f...,2022-05-16,2022-05-23,us-east-1:0298bd5a-f687-42de-be4b-0c740347ab01,us-east-1:55ce4aec-2a3e-4596-b47a-8cd4b2fee16a,us-east-1:55ce4aec-2a3e-4596-b47a-8cd4b2fee16a...,True,False,2000,8112.7400,8008,81,NaN,163,Manually terminated previously,meta


In [4]:
def build_renter_activity_table(rentals):
    
    def get_daily_active_rental_counts(rental):
        '''given a rental, return a set of data frame rows which track the count of active weekly/daily rentals each day'''
        def build_row(renter_id, date, rarity, awr, adr, rental_id, ended_reason):
            row = {'renter_id' : renter_id,
                  'date' : date,
                  'rarity' : rarity,
                  'awr' : awr,
                  'adr' : adr,
                  'rental_id' : rental_id,
                  'ended_reason' : ended_reason}
            return row


        renter_id = rental['renter_id']
        start_date = format_time(rental['created_at']).date()
        rarity = rental['degen']['background']

        try:
            ended_reason = rental['ended_reason']
        except KeyError:
            ended_reason = None
        rental_id = rental['id']
        if rental['is_active']:
            last_date = datetime.now().date()
        else:
            try:
                last_date = format_time(rental['ended_at']).date()
            except KeyError:
                print(f"key error when looking for ended_at timestamp in {rental['id']}")
        duration = (last_date-start_date).days
        rows = []
        while duration > 0:
            delta = timedelta(days=duration)
            row = build_row(renter_id=renter_id, 
                      date=start_date+delta, 
                      rarity=rarity, 
                      awr = (duration <= 7) * 1, 
                      adr = (duration > 7) * 1, 
                      rental_id=rental_id, 
                      ended_reason=ended_reason)
            rows.append(row)
            duration -= 1
        return pd.DataFrame(rows)


    def extract_rental_info_from_renters(rentals):
        '''creates a fact table about each individual renter from an input dictionary of renters
        in: renters list
        out: renter, date, rarity multi-key, with count of active daily rentals and active weekly rentals'''

        df = pd.DataFrame(columns=['renter_id', 'date', 'rarity', 'adr', 'awr', 'rental_id', 'ended_reason'])

        for rental in rentals:
            new_rows = get_daily_active_rental_counts(rental)
            df = append_df(new_rows, df)
        return df
    
    df = extract_rental_info_from_renters(rentals)
    return df

renter_activity_table = build_renter_activity_table(rentals)
renter_activity_table.head()

,renter_id,date,rarity,adr,awr,rental_id,ended_reason
0,us-east-1:56799818-79f8-441b-8e40-9453890c1537,2022-05-25,common,1,0,331#us-east-1:56799818-79f8-441b-8e40-9453890c...,Manually terminated previously
1,us-east-1:56799818-79f8-441b-8e40-9453890c1537,2022-05-24,common,1,0,331#us-east-1:56799818-79f8-441b-8e40-9453890c...,Manually terminated previously
2,us-east-1:56799818-79f8-441b-8e40-9453890c1537,2022-05-23,common,0,1,331#us-east-1:56799818-79f8-441b-8e40-9453890c...,Manually terminated previously
3,us-east-1:56799818-79f8-441b-8e40-9453890c1537,2022-05-22,common,0,1,331#us-east-1:56799818-79f8-441b-8e40-9453890c...,Manually terminated previously
4,us-east-1:56799818-79f8-441b-8e40-9453890c1537,2022-05-21,common,0,1,331#us-east-1:56799818-79f8-441b-8e40-9453890c...,Manually terminated previously


In [5]:
save_csv(renter_activity_table, 'renter_daily_activity')
save_csv(rentals_table, 'rentals')

saving at data/2022-05-27_renter_daily_activity.csv
saving at data/2022-05-27_rentals.csv


In [6]:
print('daily active renters')
pd.DataFrame(renter_activity_table.groupby('date')['renter_id'].count())

daily active renters


,renter_id
date,
2022-05-01,3
2022-05-02,3
2022-05-03,10
2022-05-04,119
2022-05-05,163
2022-05-06,213
2022-05-07,423
2022-05-08,687
2022-05-09,920


In [7]:
print('new rentals created')
pd.DataFrame(rentals_table.groupby('created_at')['id'].count())

new rentals created


,id
created_at,
2022-04-30,3
2022-05-02,7
2022-05-03,109
2022-05-04,44
2022-05-05,50
2022-05-06,210
2022-05-07,267
2022-05-08,233
2022-05-09,174


In [8]:
print('instantaneously active rentals')
rentals_table.groupby('is_active')['is_active'].count()

instantaneously active rentals


is_active
False    2525
True     1576
Name: is_active, dtype: int64